# Phase 3 — Co-mutation lookup tool (Aim 1)

**Purpose:** a clinician meets a patient with, say, lung cancer and *KRAS* G12C,
and asks: *what else is usually altered with this, how common is this exact
combination, and does it matter?* Today Jason answers by searching GENIE by hand.
This notebook does that lookup.

**Input:** a cancer type + the patient's known alteration(s) — a gene, optionally a
protein change (`p.G12C`) or a copy-number change (`AMP` / `DEL`).

**Output:**
1. `lookup()` — every partner gene seen with the query alteration: how often in
   carriers vs non-carriers, split by TMB group, with the Phase 2 test result
   attached where the pair was testable.
2. `combo_prevalence()` — how many patients carry an exact multi-gene combination.

**Inputs (Phase 1 / Phase 2 outputs):** `clinical_tidy`, `alterations_long`,
`panel_gene_coverage`, `cna_gene_panel_coverage`, `comutation_pairs`.

**Why it reads the Phase 1 tables, not just Phase 2's results:** Phase 2's gates
(gene altered in ≥ 3% of patients, ≥ 5 patients *expected* with both) exist to
make a 33,092-pair screen trustworthy — but they remove exactly the rare
combinations clinicians ask about. Here every combination is counted; Phase 2's
verdict is shown next to it where one exists.

**Not included yet:** therapy evidence (OncoKB API, needs an academic token) and an
on-demand test for rare pairs — see the last section.

## 1. Load

Same patient base as Phase 2: one sample per patient, copy-number tested only — so
"not altered" means both mutation and copy number were actually checked.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
ULTRAMUTATED_TMB = 100   # same cut-off as Phase 2 Section 6c (Campbell et al. 2017)

clinical = pd.read_parquet(PROCESSED_DIR / "clinical_tidy.parquet")
clinical = clinical[clinical["IS_REPRESENTATIVE"] & clinical["CNA_TESTED"]].set_index("SAMPLE_ID")

alterations = pd.read_parquet(
    PROCESSED_DIR / "alterations_long.parquet",
    columns=["Sample_ID", "Hugo_Symbol", "Alteration_Type", "HGVSp_Short", "CNA_Direction"])
alterations = alterations[alterations["Sample_ID"].isin(clinical.index)]

# A gene counts as "tested" only where the panel covers it for mutations AND copy number.
coverage = (pd.read_parquet(PROCESSED_DIR / "panel_gene_coverage.parquet")[["SEQ_ASSAY_ID", "Hugo_Symbol"]]
            .merge(pd.read_parquet(PROCESSED_DIR / "cna_gene_panel_coverage.parquet")))
phase2 = pd.read_parquet(PROCESSED_DIR / "comutation_pairs.parquet")

# Three TMB groups; "unknown" = panel too small to tell (< 1 Mb).
tmb_group = np.select(
    [clinical["HYPERMUTATION_STATUS"] == "not_hypermutated",
     (clinical["HYPERMUTATION_STATUS"] == "hypermutated") & (clinical["TMB_MUT_PER_MB"] < ULTRAMUTATED_TMB),
     clinical["HYPERMUTATION_STATUS"] == "hypermutated"],
    ["non-hyper", "hyper", "ultra"], default="unknown")
clinical["TMB_GROUP"] = tmb_group
print(f"{len(clinical):,} patients, {len(alterations):,} alteration rows")
print(clinical["TMB_GROUP"].value_counts().to_string())

138,206 patients, 1,055,981 alteration rows
TMB_GROUP
non-hyper    117562
hyper         15560
unknown        4605
ultra           479


## 2. Helpers

- `carriers_of()` — which patients carry the query alteration. A query is a gene
  plus an optional detail: a protein change (`"p.G12C"`), `"AMP"`, `"DEL"`, or
  `None` for any alteration of that gene.
- `tested_counts()` — for a group of patients, how many were tested for each gene
  (panel-aware denominator: a patient only counts for genes their panel covers).
- `wilson()` — 95% interval for a proportion; wide intervals flag small numbers.

In [2]:
def carriers_of(sub_alt: pd.DataFrame, gene: str, detail: str | None = None) -> set:
    hit = sub_alt["Hugo_Symbol"] == gene
    if detail in ("AMP", "DEL"):
        hit &= sub_alt["CNA_Direction"] == {"AMP": "Amplification", "DEL": "Deep_Deletion"}[detail]
    elif detail is not None:
        hit &= sub_alt["HGVSp_Short"] == detail
    return set(sub_alt.loc[hit, "Sample_ID"])


def tested_counts(patients: pd.DataFrame) -> pd.Series:
    """Per gene: how many of these patients had that gene tested."""
    per_panel = patients["SEQ_ASSAY_ID"].value_counts()
    cov = coverage[coverage["SEQ_ASSAY_ID"].isin(per_panel.index)]
    return cov["SEQ_ASSAY_ID"].map(per_panel).groupby(cov["Hugo_Symbol"]).sum()


def altered_counts(sub_alt: pd.DataFrame, patients) -> pd.Series:
    """Per gene: how many of these patients have it altered (any alteration)."""
    return sub_alt[sub_alt["Sample_ID"].isin(patients)].groupby("Hugo_Symbol")["Sample_ID"].nunique()


def wilson(k, n, z=1.96):
    k, n = np.asarray(k, float), np.maximum(np.asarray(n, float), 1)
    p, denom = k / n, 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return np.clip(centre - half, 0, 1), np.clip(centre + half, 0, 1)

## 3. `lookup()` — what else is altered with this alteration?

For each partner gene:

| column | meaning |
|---|---|
| `n_both / n_tested` | carriers of the query who also have the partner / carriers tested for the partner |
| `pct_carriers` (+ 95% CI) | how often the partner is altered **with** the query alteration |
| `pct_non_carriers` | how often it is altered in patients **without** it (tested for both genes) |
| `ratio` | `pct_carriers / pct_non_carriers` — > 1 more common together, < 1 less |
| `pct_carriers_nonhyper` / `pct_carriers_hyper` | the same, within TMB groups — raw frequencies mislead when hypermutated tumours pile up alterations |
| `phase2` | Phase 2's gene-level verdict (non-hypermutated patients): `candidate, replicated` (also confirmed in centres other than MSK), `candidate` (significant, not a copy-number artefact), `significant` (q < 0.05 only), `n.s.`, or `not tested` (below Phase 2's gates — often a rare combination) |

Phase 2's verdict is at **gene** level (e.g. *KRAS* vs *STK11*), even when the query
is a specific variant (*KRAS* G12C). `check` flags partners where this variant's own
raw ratio points the opposite way from Phase 2. Two possible reasons: the variant
behaves unlike its gene as a whole (Example 3), or the raw percentages are inflated by
heavily altered (often hypermutated) patients, which Phase 2's test adjusts for
(Example 2: *PIK3CA*). Where they disagree, trust Phase 2 for the gene and the
non-hyper column for the size of the effect.

In [3]:
def lookup(cancer_type: str, gene: str, detail: str | None = None,
           min_both: int = 3, top: int | None = 15) -> pd.DataFrame:
    ct = clinical[clinical["CANCER_TYPE"] == cancer_type]
    sub_alt = alterations[alterations["Sample_ID"].isin(ct.index)]

    # Only patients whose panel tested the query gene can be called carrier / non-carrier.
    base = ct[ct["SEQ_ASSAY_ID"].isin(coverage.loc[coverage["Hugo_Symbol"] == gene, "SEQ_ASSAY_ID"])]
    carriers = carriers_of(sub_alt, gene, detail) & set(base.index)
    car, non = base.loc[sorted(carriers)], base.drop(index=list(carriers))
    label = f"{gene} {detail or '(any alteration)'}"
    print(f"{cancer_type} | {label}: {len(car):,} of {len(base):,} tested patients ({len(car) / len(base):.1%})")
    print("  carriers by TMB group: " + ", ".join(f"{k} {v:,}" for k, v in car["TMB_GROUP"].value_counts().items()))

    out = pd.DataFrame({"n_both": altered_counts(sub_alt, car.index), "n_tested": tested_counts(car)})
    out = out.drop(index=gene, errors="ignore").dropna()
    out = out[out["n_both"] >= min_both].astype(int)
    out["pct_carriers"] = out["n_both"] / out["n_tested"]
    out["ci_low"], out["ci_high"] = wilson(out["n_both"], out["n_tested"])
    out["pct_non_carriers"] = (altered_counts(sub_alt, non.index) / tested_counts(non)).reindex(out.index).fillna(0)
    out["ratio"] = out["pct_carriers"] / out["pct_non_carriers"].where(out["pct_non_carriers"] > 0)
    for grp, col in [("non-hyper", "pct_carriers_nonhyper"), ("hyper", "pct_carriers_hyper")]:
        g = car[car["TMB_GROUP"] == grp]
        out[col] = (altered_counts(sub_alt, g.index) / tested_counts(g)).reindex(out.index)

    # Phase 2's gene-level verdict for (query gene, partner) in this cancer type.
    p2 = phase2[(phase2["Cancer_Type"] == cancer_type) & ((phase2["Gene_A"] == gene) | (phase2["Gene_B"] == gene))]
    p2 = p2.assign(partner=p2["Gene_B"].where(p2["Gene_A"] == gene, p2["Gene_A"])).set_index("partner")
    verdict = np.select([p2["interaction_candidate"] & p2["replicated"], p2["interaction_candidate"], p2["q_value"] < 0.05],
                        ["candidate, replicated", "candidate", "significant"], "n.s.")
    out["phase2"] = pd.Series(verdict, index=p2.index).reindex(out.index).fillna("not tested")
    out["phase2_direction"] = p2["direction"].reindex(out.index).where(out["phase2"] != "not tested", "")
    out["phase2_q"] = p2["q_value"].reindex(out.index)
    # Flag where the raw ratio points the other way from Phase 2's verdict. Two causes:
    # the variant behaves unlike its gene (MET amplification vs MET mutations), or the
    # raw % is inflated by heavily altered patients, which Phase 2 adjusts for.
    this_dir = np.where(out["ratio"] > 1, "co-occurring", "exclusive")
    out["check"] = np.where(out["phase2"].isin(["candidate, replicated", "candidate", "significant"])
                            & (out["phase2_direction"] != this_dir), "differs from Phase 2", "")

    out = out.sort_values("n_both", ascending=False)
    return out if top is None else out.head(top)


def show(t: pd.DataFrame) -> pd.DataFrame:
    """Readable view: percentages rounded, CI as one column."""
    v = t.copy()
    for c in ["pct_carriers", "pct_non_carriers", "pct_carriers_nonhyper", "pct_carriers_hyper"]:
        v[c] = (100 * v[c]).round(1)
    v["95% CI"] = [f"{100 * lo:.0f}-{100 * hi:.0f}" for lo, hi in zip(v["ci_low"], v["ci_high"])]
    v["ratio"] = v["ratio"].round(2)
    v["phase2_q"] = v["phase2_q"].map(lambda q: "" if pd.isna(q) else f"{q:.1e}")
    return v[["n_both", "n_tested", "pct_carriers", "95% CI", "pct_non_carriers", "ratio",
              "pct_carriers_nonhyper", "pct_carriers_hyper", "phase2", "phase2_direction", "phase2_q", "check"]]

### Example 1 — lung cancer, *KRAS* G12C

Check against the literature: about a quarter of *KRAS*-mutant lung cancers carry
*STK11* (Skoulidis et al. 2018), a combination linked to poor immunotherapy response.

In [4]:
show(lookup("Non-Small Cell Lung Cancer", "KRAS", "p.G12C"))

Non-Small Cell Lung Cancer | KRAS p.G12C: 1,881 of 16,596 tested patients (11.3%)
  carriers by TMB group: non-hyper 1,250, hyper 561, unknown 70


,n_both,n_tested,pct_carriers,95% CI,pct_non_carriers,ratio,pct_carriers_nonhyper,pct_carriers_hyper,phase2,phase2_direction,phase2_q,check
Hugo_Symbol,,,,,,,,,,,,
TP53,741,1860,39.8,38-42,55.0,0.72,31.3,56.9,"candidate, replicated",exclusive,1.5e-43,
STK11,486,1860,26.1,24-28,12.1,2.15,28.6,19.6,"candidate, replicated",co-occurring,8.2e-75,
KEAP1,353,1816,19.4,18-21,12.5,1.56,19.2,19.3,"candidate, replicated",co-occurring,6.1e-36,
ATM,269,1844,14.6,13-16,5.6,2.60,14.1,15.7,"candidate, replicated",co-occurring,2.7e-28,
RBM10,261,1667,15.7,14-17,8.9,1.75,13.8,21.6,"candidate, replicated",co-occurring,3.1e-05,
CDKN2A,258,1860,13.9,12-16,18.1,0.77,11.1,20.7,candidate,exclusive,6.1e-03,
SMARCA4,186,1860,10.0,9-11,7.6,1.31,8.6,13.0,"candidate, replicated",co-occurring,5.8e-10,
NKX2-1,175,1859,9.4,8-11,7.8,1.21,9.6,9.3,n.s.,co-occurring,7.2e-02,
CDKN2B,145,1859,7.8,7-9,10.4,0.75,7.0,10.2,candidate,exclusive,8.6e-06,


### Example 2 — colorectal cancer, *BRAF* V600E: why the TMB columns matter

Most *BRAF* V600E colorectal tumours are mismatch-repair deficient (hypermutated).
Partners like *RNF43* look strongly enriched overall; the TMB columns and Phase 2's
verdict show how much of that is hypermutation rather than an interaction: *KMT2D*
is 27% overall but 3% in non-hypermutated carriers, and Phase 2 finds no interaction.
*PIK3CA* looks enriched in the raw numbers but is flagged: in non-hypermutated
carriers it is 13% vs 19% in non-carriers, and Phase 2 calls it exclusive.

In [5]:
show(lookup("Colorectal Cancer", "BRAF", "p.V600E"))

Colorectal Cancer | BRAF p.V600E: 981 of 12,844 tested patients (7.6%)
  carriers by TMB group: non-hyper 504, hyper 440, unknown 21, ultra 16


,n_both,n_tested,pct_carriers,95% CI,pct_non_carriers,ratio,pct_carriers_nonhyper,pct_carriers_hyper,phase2,phase2_direction,phase2_q,check
Hugo_Symbol,,,,,,,,,,,,
TP53,599,978,61.2,58-64,74.9,0.82,81.3,37.7,n.s.,exclusive,7.5e-01,
RNF43,437,912,47.9,45-51,5.8,8.31,22.3,75.5,not tested,,,
APC,285,978,29.1,26-32,77.6,0.38,26.8,30.5,"candidate, replicated",exclusive,4.6e-38,
KMT2D,262,962,27.2,25-30,7.6,3.60,3.0,52.9,not tested,,,
ARID1A,245,962,25.5,23-28,9.8,2.59,4.1,50.1,n.s.,exclusive,2.0e-01,
PIK3CA,243,981,24.8,22-28,19.0,1.30,13.3,36.6,candidate,exclusive,1.6e-02,differs from Phase 2
SMAD4,223,978,22.8,20-26,17.4,1.31,31.0,13.4,"candidate, replicated",co-occurring,8.5e-03,
FBXW7,200,978,20.4,18-23,14.2,1.44,10.1,32.0,n.s.,exclusive,1.7e-01,
MSH3,182,594,30.6,27-34,4.6,6.73,NaN,54.6,not tested,,,


### Example 3 — a copy-number query: lung cancer, *MET* amplification

Why the variant matters: *EGFR* is **more** common with *MET* amplification (38% vs
24%), yet Phase 2 calls *MET*/*EGFR* **exclusive**. Phase 2 tests *MET* as a whole
gene, dominated by *MET* mutations (exon 14 skipping, a driver on its own, which
avoids *EGFR*). *MET* amplification is different: a known escape route of
*EGFR*-mutant tumours under EGFR inhibitors. The `check` column flags it.

In [6]:
show(lookup("Non-Small Cell Lung Cancer", "MET", "AMP"))

Non-Small Cell Lung Cancer | MET AMP: 376 of 16,596 tested patients (2.3%)
  carriers by TMB group: non-hyper 268, hyper 101, unknown 7


,n_both,n_tested,pct_carriers,95% CI,pct_non_carriers,ratio,pct_carriers_nonhyper,pct_carriers_hyper,phase2,phase2_direction,phase2_q,check
Hugo_Symbol,,,,,,,,,,,,
TP53,300,376,79.8,75-84,52.6,1.52,75.4,92.1,n.s.,co-occurring,8.6e-01,
EGFR,141,376,37.5,33-42,23.6,1.59,44.0,21.8,candidate,exclusive,5.0e-06,differs from Phase 2
CDKN2A,96,376,25.5,21-30,17.4,1.46,24.3,28.7,n.s.,exclusive,8.0e-01,
CDKN2B,62,376,16.5,13-21,9.9,1.66,17.2,13.9,n.s.,co-occurring,8.6e-01,
KEAP1,58,362,16.0,13-20,13.2,1.21,8.5,36.1,"candidate, replicated",exclusive,1.8e-09,differs from Phase 2
MYC,48,376,12.8,10-17,5.1,2.49,14.6,7.9,n.s.,co-occurring,5.3e-01,
BRAF,42,376,11.2,8-15,4.8,2.33,10.4,12.9,n.s.,exclusive,7.7e-01,
TERT,40,350,11.4,9-15,5.8,1.98,12.1,9.2,"candidate, replicated",co-occurring,1.7e-04,
NKX2-1,38,376,10.1,7-14,7.9,1.28,10.8,7.9,candidate,exclusive,6.8e-03,differs from Phase 2


## 4. `combo_prevalence()` — how common is this exact combination?

Counts patients carrying **all** of the listed alterations, among patients whose
panel tested **all** of the genes, overall and per TMB group. Each item is
`(gene, detail)` as in `lookup()`.

In [7]:
def combo_prevalence(cancer_type: str, combo: list[tuple[str, str | None]]) -> pd.DataFrame:
    ct = clinical[clinical["CANCER_TYPE"] == cancer_type]
    sub_alt = alterations[alterations["Sample_ID"].isin(ct.index)]
    genes = [g for g, _ in combo]
    n_cov = coverage[coverage["Hugo_Symbol"].isin(genes)].groupby("SEQ_ASSAY_ID")["Hugo_Symbol"].nunique()
    base = ct[ct["SEQ_ASSAY_ID"].isin(n_cov[n_cov == len(set(genes))].index)]
    carriers = set(base.index)
    for g, d in combo:
        carriers &= carriers_of(sub_alt, g, d)

    rows = []
    for grp, pts in [("all", base)] + [(g, base[base["TMB_GROUP"] == g]) for g in ["non-hyper", "hyper", "ultra"]]:
        k = len(carriers & set(pts.index))
        lo, hi = wilson(k, len(pts))
        rows.append({"group": grp, "n_with_combo": k, "n_tested": len(pts),
                     "pct": round(100 * k / max(len(pts), 1), 2), "95% CI": f"{100 * lo:.2f}-{100 * hi:.2f}"})
    print(f"{cancer_type} | " + " + ".join(f"{g} {d or ''}".strip() for g, d in combo))
    return pd.DataFrame(rows).set_index("group")

In [8]:
combo_prevalence("Non-Small Cell Lung Cancer", [("KRAS", "p.G12C"), ("STK11", None), ("KEAP1", None)])

Non-Small Cell Lung Cancer | KRAS p.G12C + STK11 + KEAP1


,n_with_combo,n_tested,pct,95% CI
group,,,,
all,236,15995,1.48,1.30-1.67
non-hyper,171,12008,1.42,1.23-1.65
hyper,57,3507,1.63,1.26-2.10
ultra,0,5,0.00,0.00-43.45


In [9]:
# A combination textbooks call "mutually exclusive" -- rare, but it exists, and is
# far more common in hypermutated tumours.
combo_prevalence("Colorectal Cancer", [("BRAF", "p.V600E"), ("KRAS", None)])

Colorectal Cancer | BRAF p.V600E + KRAS


,n_with_combo,n_tested,pct,95% CI
group,,,,
all,10,12844,0.08,0.04-0.14
non-hyper,2,10298,0.02,0.01-0.07
hyper,6,2024,0.30,0.14-0.65
ultra,2,123,1.63,0.45-5.74


In [10]:
combo_prevalence("Non-Small Cell Lung Cancer", [("EGFR", "p.L858R"), ("MET", "AMP")])

Non-Small Cell Lung Cancer | EGFR p.L858R + MET AMP


,n_with_combo,n_tested,pct,95% CI
group,,,,
all,49,16594,0.30,0.22-0.39
non-hyper,43,12320,0.35,0.26-0.47
hyper,6,3606,0.17,0.08-0.36
ultra,0,5,0.00,0.00-43.45


## 5. Sanity checks

- **Denominators are panel-aware:** a partner's `n_tested` is never larger than the
  number of carriers, and is smaller for genes only some panels cover.
- **Known numbers are reproduced:** *STK11* in ~26% of *KRAS* G12C lung cancers
  (published: ~25%).

In [11]:
t = lookup("Non-Small Cell Lung Cancer", "KRAS", "p.G12C", top=None)
n_carriers = len(carriers_of(alterations[alterations["Sample_ID"].isin(
    clinical.index[clinical["CANCER_TYPE"] == "Non-Small Cell Lung Cancer"])], "KRAS", "p.G12C"))
assert (t["n_tested"] <= n_carriers).all() and (t["n_both"] <= t["n_tested"]).all()
print(f"STK11 with KRAS G12C: {t.loc['STK11', 'pct_carriers']:.1%}")
print(f"partner genes seen in >= 3 carriers: {len(t):,}; tested in Phase 2: {(t['phase2'] != 'not tested').sum():,}")

Non-Small Cell Lung Cancer | KRAS p.G12C: 1,881 of 16,596 tested patients (11.3%)
  carriers by TMB group: non-hyper 1,250, hyper 561, unknown 70


STK11 with KRAS G12C: 26.1%
partner genes seen in >= 3 carriers: 586; tested in Phase 2: 28


## 6. Next steps

1. **Therapy evidence** — annotate each query alteration with OncoKB levels
   (`/annotate/mutations/byProteinChange`, needs an academic API token), plus a
   short curated list of known co-mutation modifiers (e.g. *STK11*/*KEAP1* with
   *KRAS* in lung cancer: poor immunotherapy response).
2. **On-demand test for rare pairs** — Phase 2's conditional test for a single
   queried pair, labelled exploratory (no screen-wide correction applies to one
   question).
3. **Validation across independent subsets** — split GENIE by sequencing centre and
   check the frequencies replicate.
4. **Interface** — a small web form (e.g. Streamlit) once the logic is agreed with
   Jason.
5. **Limits to state in the tool:** co-occurrence is not treatment prediction; GENIE's
   main release has no treatment or response data.